In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score

sns.set_theme(style='whitegrid')
%matplotlib inline

df = pd.read_csv('../data/processed/train.csv')

In [ ]:
# Encode metadata and fit a logistic regression — if AUC > 0.5, metadata has signal
df2 = df.dropna(subset=['age_approx', 'sex', 'anatom_site_general_challenge']).copy()
df2['sex_enc'] = LabelEncoder().fit_transform(df2['sex'])
df2['site_enc'] = LabelEncoder().fit_transform(df2['anatom_site_general_challenge'])

X = df2[['age_approx', 'sex_enc', 'site_enc']].values
y = df2['target'].values

lr = LogisticRegression(class_weight='balanced', max_iter=500)
lr.fit(X, y)
preds = lr.predict_proba(X)[:, 1]
auroc = roc_auc_score(y, preds)
print(f'Metadata-only logistic regression AUC-ROC: {auroc:.4f}')
print('(> 0.5 confirms metadata carries predictive signal beyond random chance)')

In [ ]:
# Positive rate by age group
df2['age_group'] = pd.cut(df2['age_approx'], bins=[0, 30, 45, 60, 75, 100],
                           labels=['<30', '30-45', '45-60', '60-75', '75+'])
pos_by_age = df2.groupby('age_group')['target'].mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

pos_by_age.plot(kind='bar', ax=axes[0], color='coral')
axes[0].set_title('Malignant Rate by Age Group')
axes[0].set_ylabel('Positive Fraction')
axes[0].tick_params(axis='x', rotation=0)

# Positive rate by sex
pos_by_sex = df2.groupby('sex')['target'].mean()
pos_by_sex.plot(kind='bar', ax=axes[1], color='mediumpurple')
axes[1].set_title('Malignant Rate by Sex')
axes[1].set_ylabel('Positive Fraction')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance via logistic regression coefficients
coef = pd.Series(
    np.abs(lr.coef_[0]),
    index=['age_approx', 'sex', 'anatom_site']
).sort_values(ascending=True)

coef.plot(kind='barh', color='teal')
plt.title('Metadata Feature Importance (|LR coefficient|)')
plt.xlabel('|Coefficient|')
plt.tight_layout()
plt.show()
print('Higher = more predictive signal for malignancy')